<a href="https://colab.research.google.com/github/SafaaMahbub/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5
...,...,...,...,...
395,V-18,Merch,1,12.0
396,V-01,Merch,2,24.0
397,V-10,Food,3,7.5
398,V-18,Merch,2,24.0


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
df['revenue']= df['qty'] * df['price']
total_revenue = df['revenue'].sum()
print('total revenue',total_revenue, 'total units', df['qty'].sum())
df

total revenue 8520.0 total units 783


,vendor_id,category,qty,price,revenue
0,V-10,Drink,2,24.0,48.0
1,V-18,RainGear,1,12.0,12.0
2,V-18,Drink,3,4.5,13.5
3,V-10,Food,2,12.0,24.0
4,V-18,Drink,3,7.5,22.5
...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0
396,V-01,Merch,2,24.0,48.0
397,V-10,Food,3,7.5,22.5
398,V-18,Merch,2,24.0,48.0


The total revenue is 8520 which was computed from the sum of (qty*price). The total units are 783 which is the sum of the column qty.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
by_category = (df.groupby('category')
           .agg(units=('qty', 'sum'),
                revenue=('revenue', 'sum'))
           .round(2)
           .sort_values('revenue', ascending=False))
by_category['total_share'] = (by_category['revenue']/total_revenue*100).round(2)
by_category

,units,revenue,total_share
category,,,
Food,362,4293.0,50.39
Merch,158,1771.5,20.79
Drink,178,1554.0,18.24
RainGear,85,901.5,10.58


the highest total share form this table is the food category as they make around 50% of the total revenue.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
summary2 = (df.groupby('vendor_id')
           .agg(order_count=('vendor_id','count'),
               units=('qty', 'sum'),
                average_revenue=('revenue', 'mean'))
           .round(2)
           .sort_values('average_revenue', ascending=False))
summary2

,order_count,units,average_revenue
vendor_id,,,
V-01,94,188,22.60
V-18,108,217,21.75
V-05,93,178,20.58
V-10,105,200,20.31


V-01 has the highest average order revenue at 22.60. 22.60 means that the average profit of an order from V-01 is $22.60.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
by_category.loc['Merch','total_share']
print('merch_revenue_share:', by_category.loc['Merch','total_share'])

merch_revenue_share: 20.79


The share of revenue for the category march is 20.79% which can be seen in the summary table.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
baseline_rows = len(df)
total_revenue_before = df['revenue'].sum()
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})
print(f'before: {baseline_rows} rows, ${total_revenue_before:.2f}')
joined = df.merge(vendor_names, on='vendor_id', how='left', indicator=True, validate='many_to_one')
print(f'after: {len(joined)} rows, ${joined['revenue'].sum():.2f}')
print(joined['_merge'].value_counts())

unmatched = joined[joined['_merge']=='left_only']

print(f'not matching orders',len(unmatched))
print(f'uknown vendor',unmatched['vendor_id'].unique())
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')
# TODO: merge, validate, and report the unmatched vendor

joined

before: 400 rows, $8520.00
after: 400 rows, $8520.00
_merge
both          292
left_only     108
right_only      0
Name: count, dtype: int64
not matching orders 108
uknown vendor ['V-18']


,vendor_id,category,qty,price,revenue,vendor_name,_merge
0,V-10,Drink,2,24.0,48.0,Cav Merch North,both
1,V-18,RainGear,1,12.0,12.0,Unknown vendor,left_only
2,V-18,Drink,3,4.5,13.5,Unknown vendor,left_only
3,V-10,Food,2,12.0,24.0,Cav Merch North,both
4,V-18,Drink,3,7.5,22.5,Unknown vendor,left_only
...,...,...,...,...,...,...,...
395,V-18,Merch,1,12.0,12.0,Unknown vendor,left_only
396,V-01,Merch,2,24.0,48.0,Hoos Burgers,both
397,V-10,Food,3,7.5,22.5,Cav Merch North,both
398,V-18,Merch,2,24.0,48.0,Unknown vendor,left_only


**The unmatched vendor, and what I did about it:** _..._
I chose to just replace the vendor name for unknown vendors with "Uknown Vendor" because I wanted to preserve the total revenue. Another reason on why I chose to just fill in the null values with "uknown vendor" was to make a note that for those rows, a more investigation needs to be done to find out what the actual vendor name is.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [13]:
# TODO
pivot_table = df.pivot_table(index='vendor_id',columns='category',values='revenue',aggfunc='sum',margins_name='total', margins=True)

pivot_table

category,Drink,Food,Merch,RainGear,total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
total,1554.0,4293.0,1771.5,901.5,8520.0


I chose to use the pivot table that is in the pandas library. The index are the vendor ids and the columns are the categories. I used the aggregation function sum to sum up if there are duplicated vendor ids. To add the totals for each row, margins were added in the parameters

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

**a)** Since the category food made the most revenue with 50.4% total share of the total revenue, I would tell the vendors to add more variety of food items to increase the total revenue. Since the category raingear has only a share of 10.58%, the game should be pused on a rainy day so that more raincoats are being bought to increase the revenue.

**b**)
v-18 is considered unmatched vendor which does not make sense as it basically contributed the most to the revenue. I would assume that that particular vendor is famous and known for its items. V-18 made a revenue of $2349.